In [5]:
from operator import length_hint

from langchain_classic.schema import retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from vector_store import vector_store

'''
RunnableParallel은 두 개의 Chain을 병렬로 실행할 수 있는 클래스입니다. Runnable의 하위 클래스로, 생성자에서 Chain 1과 Chain 2를 초기화하고 run 메서드에서 Chain 1과 Chain 2를 병렬로 실행합니다.
'''

from dotenv import load_dotenv
import os

load_dotenv()
model = ChatGroq(
    model = 'llama-3.1-8b-instant'
)

parser = StrOutputParser()

chain_1 = (
    ChatPromptTemplate.from_template('{topic}을 한 문장으로 요약해줘')
    |model
    |parser
)

chain_2 = (
    ChatPromptTemplate.from_template('{topic}의 실 사용 예시를 한가지만 알려줘')
    |model
    |parser
)


parallel = RunnableParallel(
    chain_1=chain_1,
    chain_2=chain_2
)

# RunnableParallel의 invoke 메서드 호출 및 결과 출력
result = parallel.invoke({
    'topic' : 'python'
})

# 결과 타입 출력
print(type(result))

# 결과 키 출력
print(result.keys())

# Chain 1 결과 출력
print('==첫번째 결과==')
print(result['chain_1'])


<class 'dict'>
dict_keys(['chain_1', 'chain_2'])
==첫번째 결과==
파이썬(Python)은 간결하고 읽기 쉬운 코드를 사용하는 고급 프로그래밍 언어로, 데이터 분석, 웹 개발, 인공지능, 게임 개발 등 다양한 분야에서 널리 사용되는 강력한 도구입니다.
==두번째 결과==
Python의 실 사용 예시는 다음과 같습니다.

**웹 스크래핑**

웹 스크래핑은 웹페이지의 정보를 추출하여 사용하는 기술입니다. Python의 `requests`와 `BeautifulSoup` 라이브러리를 사용하여 웹페이지의 HTML 정보를 추출하고, 필요한 정보를 추출하여 사용할 수 있습니다.

예를 들어, 다음은 네이버 뉴스 홈페이지의 제목을 추출하는 예제입니다.
```python
import requests
from bs4 import BeautifulSoup

# 네이버 뉴스 홈페이지의 URL
url = "https://news.naver.com/main/read.nhn?oid=news&aid=0000000000"

# requests 라이브러리를 사용하여 URL에 요청을 보냅니다.
response = requests.get(url)

# HTML 정보를 Beautiful Soup 객체로 파싱합니다.
soup = BeautifulSoup(response.content, 'html.parser')

# 뉴스 제목을 추출합니다.
title = soup.find('h3', {'class': 'title'}).text.strip()

# 추출한 제목을 출력합니다.
print(title)
```
이 예제는 네이버 뉴스 홈페이지의 제목을 추출하는 코드입니다. `requests` 라이브러리를 사용하여 URL에 요청을 보내고, `BeautifulSoup` 라이브러리를 사용하여 HTML 정보를 파싱합니다. 추출한 제목을 출력하는 코드를 작성할 수 있습니다.


In [7]:
from langchain_core.runnables import RunnableLambda

# 프로세스 장단점 Chain 생성
pros_chain = ChatPromptTemplate.from_template('{topic}의 장점 두가지') |model|parser
cons_chain = ChatPromptTemplate.from_template('{topic}의 단점 두가지')|model|parser


# Chain 딕셔너리 생성
parallel_dict={
    'pros_chain': pros_chain,
    'cons_chain':cons_chain
}

# RunnableLambda에 프로세스 장단점 Chain 결합
chain = parallel_dict | RunnableLambda(lambda  x :print(x))

# invoke 메서드 호출
result = chain.invoke({
    'topic': 'java'
})

print(result)


{'pros_chain': 'Java는 다양한 플랫폼에서 실행되는 멀티태스킹 및 객체 지향 프로그래밍(OOP) 언어입니다. Java의 장점은 다음과 같습니다.\n\n1. **플랫폼 독립성**: Java는 플랫폼 독립적인 언어로, 운영 체제(Windows, macOS, Linux 등)에 관계없이 Java 코드를 작성할 수 있습니다. Java 가상 머신(Virtual Machine, VM)을 통해 Java 코드를 운영 체제에 실행할 수 있기 때문에, Java는 여러 운영 체제에서 실행할 수 있습니다.\n2. **객체 지향 프로그래밍(OOP)**: Java는 객체 지향 프로그래밍(OOP) 언어로, 객체를 생성하고 관리할 수 있습니다. Java는 클래스, 객체, 상속, 다형성, 추상화 등 OOP의 개념을 제공하여, 프로그램을 구조화하고 유지 보수하기 쉽게 합니다.', 'cons_chain': 'Java의 단점 두 가지는 다음과 같습니다.\n\n1. **성능이 느리다**: Java는 인터프리터를 통해 실행되기 때문에 일반 C나 C++과 비교하여 성능이 느립니다. 인터프리터는 프로그램이 바이트 코드를 직접 실행할 수 있도록 하기 때문에, Java 애플리케이션을 실행하는 데 추가적인 시간과 메모리가 필요합니다.\n\n2. **런타임 오류에 취약하다**: Java는 런타임 오류에 취약합니다. Java 애플리케이션은 런타임에 오류가 발생할 수 있으며, 이러한 오류는 일반적으로 JVM이 종료되는 것을 유발합니다. Java는 런타임 오류를 감지하고 해결하기 위해 다양한 기능을 제공하므로, 이러한 문제를 완화할 수 있습니다. 다만, 이러한 기능은 추가적인 오버헤드를 유발할 수 있습니다.\n\n참고로, Java의 이러한 단점은 Java 8 이후로 일부 개선되었으며, Java 11부터는 GraalVM과 같은 Just-In-Time (JIT) 컴파일러를 사용하여 성능을 개선하고 있습니다.'}
None


In [15]:
inspect_a = RunnableLambda(lambda x:f'브랜치 A가 받는 입력 = {x}')
inspect_b = RunnableLambda(lambda x:f'브랜치 B가 받는 입력 = {x}')

parallel_inspect = RunnableParallel(
    a=inspect_a,
    b=inspect_b
)

result=parallel_inspect.invoke({
    'name': '수원',
    'location': '경기도'
})

'''
RunnableParallel은 이름 그대로 여러 작업을 병렬로 실행하는데,
이때 가장 처음에 받은 입력값을 쪼개지 않고 연결된 모든 하위 브랜치에 동일하게 전달합니다

동작 원리

- 입력 전달: invoke() 안에 넣은 딕셔너리 데이터 전체가 parallel_inspect로 들어갑니다.

 동일한 데이터 복사(Broadcast): RunnableParallel은 자신이 받은 입력 데이터를 하위 항목인 a(inspect_a)와 b(inspect_b)에 똑같이 통째로 넘겨줍니다.

- 독립적 실행:

    - 브랜치 A는 딕셔너리를 받아 문자열을 만듭니다.
    - 브랜치 B도 동일한 딕셔너리를 받아 문자열을 만듭니다.

- 결과 병합: 두 브랜치의 실행이 끝나면, 각각의 결과를 다시 a와 b라는 키(Key)로 묶어서 하나의 딕셔너리로 반환합니다.
'''

print(f'a : {result.get('a')}')
print(f'b : {result.get('b')}')


a : 브랜치 A가 받는 입력 = {'name': '수원', 'location': '경기도'}
b : 브랜치 B가 받는 입력 = {'name': '수원', 'location': '경기도'}


- 현재 방법으론 과거를 전혀 기억하지 못하는 단점이 존재함

1. 파이프라인의 Stateless(무상태성) 특성으로 인해 과거를 전혀 기억하지 못하는 단점이 존재합니다.
2. 이 문제는 챗봇과 같은 문맥이 이어져야 하는 서비스에 특히나 크게 영향을 미칩니다.
3. 개발자는 파이프라인에 메모리(Memory) 시스템을 억지로 추가해야 하며, 대화가 길어질수록 코드의 복잡도가 기하급수적으로 증가합니다.

**Solution:**

1. **Context-Aware Pipeline**: 현재 파이프라인은 상태를 기억하지 못하는 한계가 있습니다. 이를 극복하기 위해 Context-Aware Pipeline을 구현할 수 있습니다. 이Pipeline은 사용자별 대화 세션(Session ID)을 관리하고 DB에 대화 내역을 읽고 쓰는 로직을 포함합니다.
2. **Stateful Lambda Functions**: 파이프라인의 Stateless 특성으로 인해 사용자별 상태를 기억하지 못합니다. 이를 극복하기 위해 Stateful Lambda Functions을 구현할 수 있습니다. 이 함수들은 각 사용자의 상태를 기억하고, 다음 요청에서 이전 상태를 참조할 수 있도록 합니다.
3. **Database Integration**: 파이프라인의 메모리(Memory) 시스템을 DB에 통합하여, 대화 내역을 저장하고 읽어올 수 있도록 합니다.

In [16]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough.assign(
    length = RunnableLambda(lambda x:len(x['text'])),
    upper = RunnableLambda(lambda x:(x['text'].upper())),
)

result = chain.invoke({
    'text' :'hello world!'
})

print(result)

{'text': 'hello world!', 'length': 12, 'upper': 'HELLO WORLD!'}


In [18]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

chain = RunnableParallel(
    input = RunnablePassthrough(),
    process = RunnableLambda(lambda x:x['text'].upper())
)

result = chain.invoke({
     'text' :'hello world!'
})

print(result)

{'input': {'text': 'hello world!'}, 'process': 'HELLO WORLD!'}


In [21]:
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings

# 임베딩모델

embedding = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_texts(
    ['사과는 과일','자동차는 이동수단','LangChain은 프레임워크'],
    embedding
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}  # 유사도 높은 문서 3개만 가져오기 세팅
)

docs = retriever.invoke("LangChain이 뭐야?")

for doc in docs:
    print(doc.page_content)


LangChain은 프레임워크
사과는 과일
